# Kaggle setup

This notebook does exactly one thing: get the repository, at one exact
pinned commit, running on a Kaggle GPU session with the right
dependencies. It does not load any data, build any split, or define any
class name. Every notebook in this project follows that same rule, see
README.md.

Before running this, push your latest commit to GitHub and copy its
short hash into PINNED_COMMIT below. Do not leave PINNED_COMMIT as
"main", a moving target defeats the point of pinning it.

In [ ]:
import os

REPO_URL = "https://github.com/PaulOkwija/afro-fetal-net.git"
PINNED_COMMIT = "REPLACE_WITH_COMMIT_HASH"
REPO_DIR = "/kaggle/working/afro-fetal-net"

assert PINNED_COMMIT != "REPLACE_WITH_COMMIT_HASH", (
    "Set PINNED_COMMIT to a real commit hash before running this cell. "
    "A result run against a moving branch cannot be traced back to "
    "anything."
)


In [ ]:
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

subprocess.run(["git", "-C", REPO_DIR, "fetch"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", PINNED_COMMIT], check=True)

actual_commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True
).stdout.strip()
print(f"Checked out commit: {actual_commit}")
assert actual_commit.startswith(PINNED_COMMIT), (
    "Checked out commit does not match PINNED_COMMIT, something is wrong "
    "with the checkout."
)


## Install pinned dependencies

Kaggle images come with their own preinstalled package versions. We
install this project's pinned versions on top, then check the result,
rather than assuming Kaggle's defaults happen to match.

In [ ]:
os.chdir(REPO_DIR)
!pip install -q -r requirements.txt --break-system-packages 2>&1 | tail -20
!pip install -q -e . --break-system-packages


In [ ]:
!python scripts/00_check_environment.py


## Verify GPU is actually available

This is a Kaggle session, so a GPU should be attached. Fail loudly here
rather than discover three cells later that training silently ran on CPU.

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected. Check the Kaggle notebook settings, "
          "Accelerator should be set to GPU, not None.")


## Weights and Biases login

Uses a Kaggle secret named WANDB_API_KEY, added under
Add-ons > Secrets in the Kaggle notebook editor. The key itself is never
written into this notebook or into git.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

try:
    secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("WANDB_API_KEY loaded from Kaggle secrets")
except Exception as e:
    print(f"Could not load WANDB_API_KEY from Kaggle secrets: {e}")
    print("Runs in this session will fall back to print only logging, "
          "see src/fetal_ai/utils/tracking.py")


## Next steps

With the environment verified, move to the data pipeline, run from the
terminal or the next notebook, in this order:

```
python scripts/01_fetch_data.py
python scripts/02_build_manifest.py
python scripts/03_build_splits.py
python scripts/04_verify_no_leakage.py
```

Do not proceed to training until scripts/04_verify_no_leakage.py prints
PASSED.